# Indonesia GVA Data Processing — 2020 National IOT

**Source:** BPS Tabel Input-Output Indonesia 2020 — Transaksi Domestik Atas Dasar Harga Dasar (17 Produk)  
**Purpose:** Derive national sectoral GVA (NTB) and tax ratios from the 2020 IOT, to be applied to provincial PDRB time-series (2021–2024) to produce GVA estimates at basic prices.

#### Key methodological notes

1. **Provincial PDRB (ADHB) ≈ GDP at market prices.** To convert to GVA at basic prices, apply: `GVA_provincial = PDRB_provincial × (1 - tax_ratio)` where `tax_ratio = net_taxes_product / gdp_market` from this IOT.

## 1. Packages setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Data Path

In [2]:
io_file_path = r"C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\00_base_data\io_idn_2020.csv"

## 3. Extract and Load Data

The CSV has 3 rows of header metadata before the actual column headers. We skip them and parse using positional column indices, not header names, to avoid ambiguity from BPS's wide format.

In [5]:
df_io_raw = pd.read_csv(
    io_file_path,
    header=None,
    skiprows=3,
    encoding='utf-8'
)

df_io_raw.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148
0,NaN,Kode,Produk,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,1800,3011,3012,3020,3030,3040,3050,3060,3090,3100,4011,4012,4013,4019,5011,5012,5013,5019,6090,7000,8000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,"Pertanian, Kehutanan, dan Perikanan",Pertambangan dan Penggalian,Industri Pengolahan,Pengadaan Listrik dan Gas,"Pengadaan Air, Pengelolaan Sampah, Limbah dan ...",Konstruksi,Perdagangan Besar dan Eceran; Reparasi Mobil d...,Transportasi dan Pergudangan,Penyediaan Akomodasi dan Makan Minum,Informasi dan Komunikasi,Jasa Keuangan dan Asuransi,Real Estate,Jasa Perusahaan,"Administrasi Pemerintahan, Pertahanan dan Jami...",Jasa Pendidikan,Jasa Kesehatan dan Kegiatan Sosial,Jasa lainnya,Total Permintaan Antara,Konsumsi Rumah Tangga,Konsumsi LNPRT,Konsumsi Pemerintah,Pembentukan Modal Tetap Bruto,Perubahan Inventori,Ekspor Barang (F.o.b),Ekspor Jasa,Total Permintaan Akhir,Total Permintaan,Impor Barang (c.i.f),Impor Jasa,Adjustment (i.f),Total Impor,Marjin Perdagangan Besar,Marjin Perdagangan Eceran,Biaya Pengangkutan,Total Marjin Perdagangan dan Biaya Pengangkutan,Total Pajak dikurang Subsidi atas Produk,Output Domestik Harga Dasar,Total Penyediaan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1,"Pertanian, Kehutanan, dan Perikanan","177,950,421","1,924","1,187,578,694","11,265","13,131","79,973,220","2,254,959","3,671,730","140,106,931","88,151","3,577",895,"122,662","2,148,319","1,006,884","14,967,609","7,365,526","1,617,265,898","680,859,607",-,-,"218,633,618","11,338,690","55,664,465",-,"966,496,380","2,583,762,278",-,-,-,-,-,-,-,-,-,"2,583,762,278","2,583,762,278",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2,Pertambangan dan Penggalian,"98,285","137,956,001","522,219,053","191,592,139","192,564","234,035,063","34,027","2,554,579",452,-,-,-,"5,960","2,773,179",325,"66,778",-,"1,091,528,405","392,584",-,-,"16,080,119","23,309,241","408,926,459",-,"448,708,403","1,540,236,808",-,-,-,-,-,-,-,-,-,"1,540,236,808","1,540,236,808",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

## 4. Cleaning helpers

In [4]:
SECTOR_COLS = list(range(3, 20))  # positional indices for the 17 industry columns

INDUSTRY_NAMES = [
    'Pertanian, Kehutanan, dan Perikanan',
    'Pertambangan dan Penggalian',
    'Industri Pengolahan',
    'Pengadaan Listrik dan Gas',
    'Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang',
    'Konstruksi',
    'Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor',
    'Transportasi dan Pergudangan',
    'Penyediaan Akomodasi dan Makan Minum',
    'Informasi dan Komunikasi',
    'Jasa Keuangan dan Asuransi',
    'Real Estate',
    'Jasa Perusahaan',
    'Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib',
    'Jasa Pendidikan',
    'Jasa Kesehatan dan Kegiatan Sosial',
    'Jasa lainnya',
]

def clean_numeric(v):
    """Convert BPS-formatted string values to float.
    Handles: commas as thousands separators, '-' as zero, leading/trailing spaces.
    """
    s = str(v).strip().replace(',', '').replace(' ', '')
    if s in ['-', '', 'nan', 'None']:
        return 0.0
    return float(s)

def get_iot_row(df, kode):
    """Extract the 17 sector values for a given row code.
    Uses column position (3:20), not header names, for robustness.
    Raises ValueError if code not found or ambiguous.
    """
    matches = df[df.iloc[:, 1].astype(str).str.strip() == str(kode)]
    if len(matches) == 0:
        raise ValueError(f"Row code '{kode}' not found in IOT")
    if len(matches) > 1:
        raise ValueError(f"Row code '{kode}' matched {len(matches)} rows — ambiguous")
    return [clean_numeric(matches.iloc[0, c]) for c in SECTOR_COLS]

## 5. Extract GVA and tax ratios from IOT

We read the following rows by Kode:

| Kode | Label | Role |
|------|-------|------|
| 2090 | Total Input Primer | **GVA at basic prices** (NTB) |
| 2010 | Kompensasi Tenaga Kerja | GVA component: labour |
| 2020 | Surplus Usaha Bruto | GVA component: operating surplus |
| 2030 | Pajak dikurang subsidi lainnya atas produksi | GVA component: other taxes on production |
| 1950 | Pajak dikurang subsidi atas produk | bridge to GDP at market prices |
| 1900 | Total Input Antara | Domestic intermediate consumption (reference only) |
| 2000 | Total Konsumsi Antara Impor | Imported intermediate consumption (reference only) |
| 2100 | Total Input | Total output at basic prices |

In [6]:
# --- GVA and its components ---
gva_basic       = get_iot_row(df_io_raw, '2090')  # Total Input Primer = NTB (Nilai Tambah Bruto)
labour_comp     = get_iot_row(df_io_raw, '2010')  # Kompensasi Tenaga Kerja
surplus_usaha   = get_iot_row(df_io_raw, '2020')  # Surplus Usaha Bruto
other_taxes_prod = get_iot_row(df_io_raw, '2030') # Pajak dikurang subsidi lainnya atas produksi

# --- Tax on products (bridges GVA to GDP at market prices) ---
net_taxes_prod  = get_iot_row(df_io_raw, '1950')  # Net Tax (Pajak dikurang subsidi atas produk)

# --- Reference rows (for verification and documentation) ---
intermediate_dom = get_iot_row(df_io_raw, '1900') # Domestic intermediate inputs
intermediate_imp = get_iot_row(df_io_raw, '2000') # Imported intermediate inputs
total_output     = get_iot_row(df_io_raw, '2100') # Total output (= total input by sector)

## 6. Build the macro dataframe

In [7]:
df_macro = pd.DataFrame({
    'industry_name'    : INDUSTRY_NAMES,
    # GVA components (all included in gva_basic)
    'labour_comp'      : labour_comp,
    'surplus_usaha'    : surplus_usaha,
    'other_taxes_prod' : other_taxes_prod,   # taxes on production — inside GVA
    # GVA at basic prices
    'gva_basic'        : gva_basic,
    # Tax on products; outside GVA, bridges to GDP
    'net_taxes_product': net_taxes_prod,
    # Reference
    'intermediate_dom' : intermediate_dom,
    'intermediate_imp' : intermediate_imp,
    'total_output'     : total_output,
})

# GDP at market prices = GVA at basic prices + net taxes on products
df_macro['gdp_market'] = df_macro['gva_basic'] + df_macro['net_taxes_product']

# Tax ratio: proportion of GDP at market prices that is net taxes on products
# Used to convert provincial PDRB (= GDP at market prices) -> GVA at basic prices
# Formula: GVA_provincial = PDRB_provincial * (1 - tax_ratio)
df_macro['tax_ratio'] = df_macro['net_taxes_product'] / df_macro['gdp_market']

# GVA share of GDP at market prices (= 1 - tax_ratio; useful for the conversion)
df_macro['gva_share_of_gdp'] = df_macro['gva_basic'] / df_macro['gdp_market']

# Sectoral GVA shares (for reference)
total_gva = df_macro['gva_basic'].sum()
df_macro['gva_share_pct'] = df_macro['gva_basic'] / total_gva * 100

df_macro[['industry_name', 'gva_basic', 'gdp_market',
          'net_taxes_product', 'tax_ratio', 'gva_share_pct']].head(17)

,industry_name,gva_basic,gdp_market,net_taxes_product,tax_ratio,gva_share_pct
0,"Pertanian, Kehutanan, dan Perikanan","2,050,855,052.0000","2,024,059,473.0000","-26,795,579.0000",-0.0132,13.6276
1,Pertambangan dan Penggalian,"979,843,647.0000","984,611,072.0000","4,767,425.0000",0.0048,6.5109
2,Industri Pengolahan,"3,141,793,316.0000","3,197,990,027.0000","56,196,711.0000",0.0176,20.8768
3,Pengadaan Listrik dan Gas,"160,374,748.0000","161,510,869.0000","1,136,121.0000",0.0070,1.0657
4,"Pengadaan Air, Pengelolaan Sampah, Limbah dan ...","43,676,775.0000","43,912,562.0000","235,787.0000",0.0054,0.2902
5,Konstruksi,"1,338,833,240.0000","1,368,802,974.0000","29,969,734.0000",0.0219,8.8964
6,Perdagangan Besar dan Eceran; Reparasi Mobil d...,"2,056,383,019.0000","2,063,086,646.0000","6,703,627.0000",0.0032,13.6644
7,Transportasi dan Pergudangan,"717,419,240.0000","728,334,981.0000","10,915,741.0000",0.0150,4.7672
8,Penyediaan Akomodasi dan Makan Minum,"521,257,226.0000","527,090,786.0000","5,833,560.0000",0.0111,3.4637
9,Informasi dan Komunikasi,"695,827,831.0000","697,532,837.0000","1,705,006.0000",0.0024,4.6237


# 7. Save out

In [15]:
df_macro.to_csv(
    r"C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data\02_01_iot_2020_gva_ratios.csv",
    index=False
)